# 17회차 — 파생변수 설계

전 생애 더위 비율, 사육 효율, 사육밀도, 상호작용 변수 생성.
회귀용 원-핫(drop_first=True, 정수형). 분류용 Target Encoding은 18회차.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

In [2]:
df = pd.read_csv("../../data/processed/3_eda/step10_spatial.csv",
                 encoding="utf-8-sig", low_memory=False)
for c in ["ABATT_DATE","BIRTH_YMD","JUDGE_DATE"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
print(f"데이터: {df.shape}")

데이터: (2408699, 56)


## 1. 시간·효율 파생변수

- fattening_days: 사육 일수 (도축일 - 출생일)
- daily_gain: 일당 증체 (도체중 / 사육일수)
- Age_squared: 사육월령 제곱 (비선형 효과)

In [3]:
df["fattening_days"] = (df["ABATT_DATE"] - df["BIRTH_YMD"]).dt.days
df["daily_gain"]     = df["WEIGHT"] / df["fattening_days"]
df["Age_squared"]    = df["AGE"] ** 2
print(f"fattening_days 0인 행: {(df['fattening_days']==0).sum()}, "
      f"daily_gain inf: {np.isinf(df['daily_gain']).sum()}")
print(df[["fattening_days","daily_gain","Age_squared"]].describe().round(2))

fattening_days 0인 행: 0, daily_gain inf: 0
       fattening_days  daily_gain  Age_squared
count      2408699.00  2408699.00   2408699.00
mean          1280.30        0.39      2290.50
std            665.34        0.15      3297.57
min              2.00        0.02         1.00
25%            921.00        0.26       961.00
50%            992.00        0.43      1089.00
75%           1413.00        0.51      2209.00
max          10470.00        1.67    118336.00


## 2. 더위 비율 변수 (15회차에서 생성, 없으면 재생성)

In [4]:
if "ratio_고온" not in df.columns:
    df["ratio_고온"]   = (df["days_주의"]+df["days_경고"]+df["days_위험"]) / df["days_total"]
    df["ratio_강더위"] = (df["days_경고"]+df["days_위험"]) / df["days_total"]
    df["ratio_위험"]   = df["days_위험"] / df["days_total"]
print(df[["ratio_고온","ratio_강더위","ratio_위험"]].describe().round(3))

          ratio_고온    ratio_강더위     ratio_위험
count  2408699.000  2408699.000  2408699.000
mean         0.415        0.274        0.032
std          0.050        0.048        0.023
min          0.000        0.000        0.000
25%          0.382        0.242        0.015
50%          0.414        0.274        0.028
75%          0.449        0.306        0.046
max          1.000        1.000        0.333


## 3. 임계값 변수 (도메인 기준 더미)

- age_optimal: 28~32개월 (한우 등급 안정 구간)
- heat_high: 고온노출 상위 25%

In [5]:
df["age_optimal"] = ((df["AGE"] >= 28) & (df["AGE"] <= 32)).astype(int)
heat_q75 = df["ratio_고온"].quantile(0.75)
df["heat_high"] = (df["ratio_고온"] >= heat_q75).astype(int)
print("임계값 더미 비율:")
print(df[["age_optimal","heat_high"]].mean().round(3))

임계값 더미 비율:
age_optimal    0.397
heat_high      0.250
dtype: float64


## 4. 농장 파생변수

- density: 사육밀도 (사육두수 / 면적)
- death_rate: 폐사율 (폐사두수 / 사육두수) — 대략치
- 농장규모: 사육두수 구간

※ 한계: C2025와 도축연도가 안 맞을 수 있고, death_count는 농장 전체·기간 불명이라
   death_rate는 대략적 지표. AREA 결측 35% → density도 결측 많음. 모두 NaN 유지.

In [6]:
# 사육밀도 — AREA 0/결측은 NaN
df["density"] = df["C2025"] / df["AREA"]
df["density"] = df["density"].replace([np.inf, -np.inf], np.nan)

# 폐사율(대략) — 분모 0/결측은 NaN
df["death_rate"] = df["death_count"] / df["C2025"]
df["death_rate"] = df["death_rate"].replace([np.inf, -np.inf], np.nan)

# 농장 규모
df["farm_size"] = pd.cut(df["C2025"], bins=[0, 50, 200, 500, np.inf],
                         labels=["소규모","중규모","대규모","초대규모"])
print(f"density 결측 {df['density'].isnull().sum()}, "
      f"death_rate 결측 {df['death_rate'].isnull().sum()}")
print(df["farm_size"].value_counts(dropna=False))

density 결측 911046, death_rate 결측 588967
farm_size
중규모     1119219
소규모      710117
대규모      347921
NaN      151741
초대규모      79701
Name: count, dtype: int64


## 5. 상호작용 변수

- density_x_heat: 사육밀도 × 고온노출 (밀집+더위 시너지)
※ 두 변수 중 하나라도 NaN이면 결과도 NaN — 유지.

In [7]:
df["density_x_heat"] = df["density"] * df["ratio_고온"]
print(df["density_x_heat"].describe().round(3))

count    1497653.000
mean           0.029
std            0.047
min            0.000
25%            0.016
50%            0.026
75%            0.038
max            5.476
Name: density_x_heat, dtype: float64


## 6. 혈통 빈도 인코딩 (Frequency Encoding)

혈통 ID(해시)는 그대로 못 쓴다. "이 부계가 몇 번 등장하나"로 바꾼다.
더미 해시·결측도 하나의 빈도로 처리됨. Target Encoding은 18회차(누수 방지).

In [8]:
lineage_cols = [c for c in ["KPN_NO","FATHER_CATTLE_NO","MOTHER_ANIMAL_NO"]
                if c in df.columns]
for col in lineage_cols:
    freq = df[col].value_counts()
    df[f"{col}_freq"] = df[col].map(freq)
print("혈통 빈도 변수 추가:")
print(df[[c+"_freq" for c in lineage_cols]].describe().round(0))

혈통 빈도 변수 추가:
       KPN_NO_freq  FATHER_CATTLE_NO_freq  MOTHER_ANIMAL_NO_freq
count    1482476.0              1486046.0              1486046.0
mean       13034.0                13052.0                 4619.0
std        17658.0                17754.0                18973.0
min            1.0                    1.0                    1.0
25%         4117.0                 4107.0                    1.0
50%         9161.0                 9161.0                    1.0
75%        14829.0                14829.0                    2.0
max        82251.0                82617.0                82748.0


## 7. 범주형 원-핫 (회귀용, drop_first=True, 정수형)

시도·계절·성별을 원-핫. drop_first=True로 더미 함정 방지.
※ .astype(int)로 정수 변환 — get_dummies 기본 bool은 OLS(statsmodels)에서 호환 문제 소지.
※ 회귀(OLS) 전용. LightGBM 분류는 18회차 Target Encoding/범주형.

In [9]:
dummy_specs = [
    ("sido", "sido"),
    ("abatt_season", "abatt_season"),
    ("birth_season", "birth_season"),
    ("JUDGE_SEX", "sex"),
]
for col, prefix in dummy_specs:
    if col in df.columns:
        d = pd.get_dummies(df[col], prefix=prefix, drop_first=True).astype(int)
        df = pd.concat([df, d], axis=1)

new_dummies = [c for c in df.columns
               if c.startswith(("sido_","abatt_season_","birth_season_","sex_"))]
print(f"원-핫 더미 {len(new_dummies)}개 추가 (dtype: {df[new_dummies[0]].dtype if new_dummies else 'N/A'})")
print(new_dummies)

원-핫 더미 24개 추가 (dtype: int64)
['sido_경기도', 'sido_경상남도', 'sido_경상북도', 'sido_광주광역시', 'sido_대구광역시', 'sido_대전광역시', 'sido_부산광역시', 'sido_서울특별시', 'sido_세종특별자치시', 'sido_울산광역시', 'sido_인천광역시', 'sido_전라남도', 'sido_전북특별자치도', 'sido_제주특별자치도', 'sido_충청남도', 'sido_충청북도', 'abatt_season_겨울', 'abatt_season_봄', 'abatt_season_여름', 'birth_season_겨울', 'birth_season_봄', 'birth_season_여름', 'sex_수', 'sex_암']


## 8. 무한대 점검 (NaN은 유지)

0으로 나눠 생긴 무한대(inf)만 NaN으로 바꾼다.
결측은 채우지 않는다 — LightGBM이 NaN을 그대로 처리하므로.

In [10]:
num_df = df.select_dtypes(include=[np.number])
inf_cols = np.isinf(num_df).sum()
inf_cols = inf_cols[inf_cols > 0]
if len(inf_cols) > 0:
    print("무한대 발견 → NaN 처리:")
    print(inf_cols)
    df = df.replace([np.inf, -np.inf], np.nan)
else:
    print("무한대 없음")

missing = df.isnull().sum()
print("\n결측 상위 10 (NaN 유지):")
print(missing[missing > 0].sort_values(ascending=False).head(10))

무한대 없음



결측 상위 10 (NaN 유지):
KPN_NO                   926223
KPN_NO_freq              926223
F_GMOTHER_ANIMAL_NO      922653
FATHER_CATTLE_NO_freq    922653
M_GFATHER_CATTLE_NO      922653
M_GMOTHER_ANIMAL_NO      922653
F_GFATHER_CATTLE_NO      922653
MOTHER_ANIMAL_NO         922653
FATHER_CATTLE_NO         922653
MOTHER_ANIMAL_NO_freq    922653
dtype: int64


## 9. 만들어진 변수 정리

In [11]:
categories = {
    # ★ WEIGHT·AGE는 test 13개 컬럼에 포함 → 트랙1(제출용) 피처로 사용 가능
    "체격 원본 (트랙1·2 공용)": ["WEIGHT", "AGE"],
    # ★ 육질 8개 + WGRADE는 test에 없음 → 도축 후 측정값 → 트랙2 분석 전용
    "육질 형질 (트랙2 전용)": ["BACKFAT","REA","WINDEX","INSFAT",
                              "YUKSAK","FATSAK","TISSUE","GROWTH"],
    "원본 가격 (트랙2 전용)": ["COST_AMT"],
    "기상 일수":   ["days_total","days_양호","days_주의","days_경고","days_위험"],
    "기상 비율":   ["ratio_고온","ratio_강더위","ratio_위험"],
    "기상 평균":   ["rn_day_mean","ws_davg_mean","ta_min_mean"],
    "시간":       ["fattening_days","Age_squared","abatt_year","abatt_month",
                  "abatt_quarter","birth_year","birth_month"],
    # ★ daily_gain = WEIGHT / fattening_days → WEIGHT가 test에 있으므로 트랙1 사용 가능
    "효율/임계":   ["daily_gain","age_optimal","heat_high"],
    "농장":       ["C2023","C2024","C2025","AREA","death_count",
                  "density","death_rate"],
    "상호작용":   ["density_x_heat"],
    "혈통 빈도":   [c for c in df.columns if c.endswith("_freq")],
    "지역 더미":   [c for c in df.columns if c.startswith("sido_")],
    "계절 더미":   [c for c in df.columns if c.startswith(("abatt_season_","birth_season_"))],
    "성별 더미":   [c for c in df.columns if c.startswith("sex_")],
}
for cat, cols in categories.items():
    n = len([c for c in cols if c in df.columns])
    print(f"{cat:30s}: {n:3d}개")
print(f"\n전체 컬럼 수: {df.shape[1]}")
print("→ 트랙1: WEIGHT·AGE·기상·시간·농장·파생 변수 사용 가능")
print("→ 트랙2: 육질 형질·가격 추가 사용 (도축 후 측정, 제출 불가)")
print("→ 분류 최종 피처 선택·Target Encoding은 18회차에서")

체격 원본 (트랙1·2 공용)              :   2개
육질 형질 (트랙2 전용)                :   8개
원본 가격 (트랙2 전용)                :   1개
기상 일수                         :   5개
기상 비율                         :   3개
기상 평균                         :   3개
시간                            :   7개
효율/임계                         :   3개
농장                            :   7개
상호작용                          :   1개
혈통 빈도                         :   3개
지역 더미                         :  16개
계절 더미                         :   6개
성별 더미                         :   2개

전체 컬럼 수: 92
→ 트랙1: WEIGHT·AGE·기상·시간·농장·파생 변수 사용 가능
→ 트랙2: 육질 형질·가격 추가 사용 (도축 후 측정, 제출 불가)
→ 분류 최종 피처 선택·Target Encoding은 18회차에서


## 10. 저장

In [12]:
df.to_csv("../../data/processed/3_eda/step11_features.csv",
          index=False, encoding="utf-8-sig")
print(f"저장 완료: step11_features.csv {df.shape}")

저장 완료: step11_features.csv (2408699, 92)
